# FB15k-237: preprocessing

Turns the official FB15k-237 **triple split** and the class labels of Jain et al. 2021 ("Do Embeddings Actually Capture Knowledge Graph Semantics?") into the files the embedding and classification work consumes, in `data/processed/fb15k-237/`:

- `train.tsv`, `valid.tsv`, `test.tsv`: the official triple split, `head<TAB>relation<TAB>tail`, no header, content and row order unchanged.
- `entity_classes.tsv`: one row per (entity, class) pair, restricted to entities that occur in the triples.
- `entity_split.tsv`: for every **experiment**, the 80/20 **entity split** of the entities that have at least one class of that experiment.

The entity split rules are copied from the authors' `classification.ipynb` (commit `2d93782`). Class labels are used as-is, without a WordNet closure; the closure is only *reported* in section 3. Terms follow `GLOSSARY.md`.

In [1]:
from pathlib import Path
import csv
import hashlib
import re
import urllib.request

import nltk
import pandas as pd
from IPython.display import display
from nltk.corpus import wordnet as wn
from sklearn.model_selection import train_test_split

DATASET = 'fb15k-237'
MIN_CLASS_SIZE = 40
TEST_SIZE = 0.2
RANDOM_STATE = 37

HF = 'https://huggingface.co/datasets/KGraph/FB15k-237/resolve/c7368ccc03358758270dbf9e475222444d19926b/data'
AUTHORS = ('https://raw.githubusercontent.com/nitishajain/KGESemanticAnalysis/'
           '2d93782b55e169ec1f5e79c9c6245b709b44c86b')
SOURCES = {  # file name -> (url, pinned SHA256)
    'train.txt': (f'{HF}/train.txt', '6e4c2782169af21e9743f3b1d200886f5d595bf6bc504ec1351720949c5cdfae'),
    'valid.txt': (f'{HF}/valid.txt', '749cbe9d923bac7b9354da5614ecfed2e0220256d442c3e04a6b303db1f273d9'),
    'test.txt': (f'{HF}/test.txt', 'e2e35e8e6113de220140b6f44dc71a5207b0fc6872d575e874aefe13259b655b'),
    'freebaseTypes.tsv': (f'{AUTHORS}/data/freebase/freebaseTypes.tsv',
                          'd21acba61828b8a773860047be2fd23692accc36b10001eaf5c40a401e57cb30'),
}
ROW_COUNTS = {'train': 272_115, 'valid': 17_535, 'test': 20_466}

# Experiments, verbatim from the authors' classification.ipynb (class IDs in YAGO form).
EXPERIMENTS = {
    'Level-1': ['wordnet_person_100007846', 'wordnet_organization_108008335',
                'wordnet_body_of_water_109225146', 'wordnet_product_104007894'],
    'Level-2-Organizations': ['wordnet_musical_organization_108246613', 'wordnet_party_108256968',
                              'wordnet_enterprise_108056231', 'wordnet_nongovernmental_organization_108009834'],
    'Level-2-Persons': ['wordnet_artist_109812338', 'wordnet_officeholder_110371450',
                        'wordnet_writer_110794014', 'wordnet_scientist_110560637',
                        'wordnet_politician_110450303'],
    'Level-3-Artists': ['wordnet_painter_110391653', 'wordnet_sculptor_110566072',
                        'wordnet_photographer_110426749', 'wordnet_illustrator_109812068',
                        'wordnet_printmaker_110475687'],
}

candidates = [Path.cwd(), Path.cwd().parent]
REPO = next((p for p in candidates if (p / 'data' / 'fb15k-237_train.tsv').is_file()), None)
if REPO is None:
    raise FileNotFoundError('Run from the repository root or the data/ directory.')
EXTERNAL = REPO / 'data' / 'external' / DATASET
OUT = REPO / 'data' / 'processed' / DATASET
COMMITTED_TRAIN = REPO / 'data' / 'fb15k-237_train.tsv'

## 1. Download and verify the sources

Every file is pinned to a commit and its SHA256 is asserted (in the pinned Hugging Face commit the train file has CRLF line endings, valid and test have LF; the processed files are all written with LF). The class-label file's SHA256 was not given in the spec; the value above is the one observed for the pinned commit of the authors' repository and is now pinned.

In [2]:
EXTERNAL.mkdir(parents=True, exist_ok=True)
for name, (url, digest) in SOURCES.items():
    path = EXTERNAL / name
    if not path.is_file() or hashlib.sha256(path.read_bytes()).hexdigest() != digest:
        req = urllib.request.Request(url, headers={'User-Agent': 'preprocessing-notebook'})
        with urllib.request.urlopen(req, timeout=120) as resp:
            path.write_bytes(resp.read())
    assert hashlib.sha256(path.read_bytes()).hexdigest() == digest, f'{name}: SHA256 mismatch'
    print(f'{name:20s} {path.stat().st_size:>10,d} bytes  sha256 ok')

train.txt            21,277,292 bytes  sha256 ok
valid.txt             1,285,566 bytes  sha256 ok
test.txt              1,499,033 bytes  sha256 ok
freebaseTypes.tsv     7,980,767 bytes  sha256 ok


## 2. Triple split

The three files are read and written back verbatim: no deduplication, no self-loop removal, no relation merging. Only the line terminator is normalised to LF.

In [3]:
def read_triples(path):
    return pd.read_csv(path, sep='\t', header=None, names=['head', 'relation', 'tail'], dtype='string',
                       keep_default_na=False, quoting=csv.QUOTE_NONE, encoding='utf-8')

OUT.mkdir(parents=True, exist_ok=True)
triples = {}
for split in ['train', 'valid', 'test']:
    triples[split] = read_triples(EXTERNAL / f'{split}.txt')
    triples[split].to_csv(OUT / f'{split}.tsv', sep='\t', header=False, index=False,
                          lineterminator='\n', quoting=csv.QUOTE_NONE, encoding='utf-8')
dataset_entities_df = pd.concat([t[c] for t in triples.values() for c in ['head', 'tail']])
ENTITIES = sorted(set(dataset_entities_df))
display(pd.DataFrame({'triples': {k: len(v) for k, v in triples.items()}}))
print(f'{len(ENTITIES):,} entities in train+valid+test')

,triples
train,272115
valid,17535
test,20466


14,541 entities in train+valid+test


## 3. Class labels

`freebaseTypes.tsv` maps Freebase entities to WordNet classes (`/m/xxx<TAB>wordnet_class_id`, no header). It is used **as-is**; only entities that occur in the triples are kept, written sorted and deduplicated.

In [4]:
labels = pd.read_csv(EXTERNAL / 'freebaseTypes.tsv', sep='\t', header=None, names=['entity', 'class'],
                     dtype='string', keep_default_na=False, quoting=csv.QUOTE_NONE, encoding='utf-8')
entity_set = set(ENTITIES)
in_dataset = labels['entity'].isin(entity_set)
entity_classes = (labels[in_dataset].drop_duplicates().sort_values(['entity', 'class'])
                  .reset_index(drop=True))
entity_classes.to_csv(OUT / 'entity_classes.tsv', sep='\t', header=True, index=False,
                      lineterminator='\n', quoting=csv.QUOTE_NONE, encoding='utf-8')
display(pd.Series({
    'rows in freebaseTypes.tsv': len(labels),
    'rows whose entity is not in the triples (dropped)': int((~in_dataset).sum()),
    'entity-class pairs kept': len(entity_classes),
    'dataset entities with >= 1 class': entity_classes['entity'].nunique(),
    'dataset entities total': len(ENTITIES),
    'coverage': round(entity_classes['entity'].nunique() / len(ENTITIES), 4),
    'distinct classes': entity_classes['class'].nunique(),
}, name='value').to_frame())

,value
rows in freebaseTypes.tsv,212270.0000
rows whose entity is not in the triples (dropped),0.0000
entity-class pairs kept,212270.0000
dataset entities with >= 1 class,12615.0000
dataset entities total,14541.0000
coverage,0.8675
distinct classes,2260.0000


### 3.1 What a WordNet closure would add (reported, not applied)

The labels are not guaranteed to be transitively closed. For information only: map each class ID to its WordNet 3.0 synset by offset (`wordnet_<lemma>_1<8-digit offset>`), take all ancestors over `hypernyms() + instance_hypernyms()`, and count how many (entity, experiment class) memberships that would add. Experiment classes are matched by offset.

In [5]:
nltk.download('wordnet', quiet=True)
assert wn.get_version() == '3.0', wn.get_version()

def offset_of(class_id):
    m = re.search(r'_1(\d{8})$', class_id)
    return int(m.group(1)) if m else None

ancestor_cache = {}
def ancestors(offset):
    if offset not in ancestor_cache:
        try:
            syn = wn.synset_from_pos_and_offset('n', offset)
            ancestor_cache[offset] = {s.offset() for s in syn.closure(lambda s: s.hypernyms() + s.instance_hypernyms())}
        except Exception:
            ancestor_cache[offset] = None
    return ancestor_cache[offset]

all_classes = sorted(entity_classes['class'].unique())
unresolved = [c for c in all_classes if offset_of(c) is None or ancestors(offset_of(c)) is None]
print(f'{len(all_classes)} distinct classes, {len(unresolved)} not resolvable in WordNet 3.0', unresolved[:5])

entity_offsets = {}
for e, c in zip(entity_classes['entity'], entity_classes['class']):
    entity_offsets.setdefault(e, set()).add(offset_of(c))
rows = []
for experiment, classes in EXPERIMENTS.items():
    for c in classes:
        off = offset_of(c)
        raw_members = {e for e, offs in entity_offsets.items() if off in offs}
        closed_members = {e for e, offs in entity_offsets.items()
                          if off in offs or any(o is not None and ancestors(o) and off in ancestors(o) for o in offs)}
        rows.append({'experiment': experiment, 'class': c, 'raw': len(raw_members),
                     'with closure': len(closed_members), 'would add': len(closed_members) - len(raw_members),
                     'kept raw (>=40)': len(raw_members) >= MIN_CLASS_SIZE,
                     'kept with closure': len(closed_members) >= MIN_CLASS_SIZE})
closure_report = pd.DataFrame(rows)
display(closure_report)
print('memberships a closure would add to the experiment classes:', int(closure_report['would add'].sum()))

2260 distinct classes, 0 not resolvable in WordNet 3.0 []


,experiment,class,raw,with closure,would add,kept raw (>=40),kept with closure
0,Level-1,wordnet_person_100007846,4446,4446,0,True,True
1,Level-1,wordnet_organization_108008335,2360,2360,0,True,True
2,Level-1,wordnet_body_of_water_109225146,24,24,0,False,False
3,Level-1,wordnet_product_104007894,1593,1593,0,True,True
4,Level-2-Organizations,wordnet_musical_organization_108246613,137,137,0,True,True
5,Level-2-Organizations,wordnet_party_108256968,42,42,0,True,True
6,Level-2-Organizations,wordnet_enterprise_108056231,170,170,0,True,True
7,Level-2-Organizations,wordnet_nongovernmental_organization_108009834,82,82,0,True,True
8,Level-2-Persons,wordnet_artist_109812338,1892,1892,0,True,True
9,Level-2-Persons,wordnet_officeholder_110371450,108,108,0,True,True


memberships a closure would add to the experiment classes: 0


## 4. Entity split

Per experiment: candidates are all dataset entities sorted by name; experiment classes with fewer than 40 members are dropped; entities with at least one remaining class are kept; `train_test_split(entities, test_size=0.2, random_state=37)`, not stratified.

In [6]:
members = entity_classes.groupby('class')['entity'].apply(set).to_dict()
class_sets = entity_classes.groupby('entity')['class'].apply(set).to_dict()

split_rows, summary = [], []
for experiment, classes in EXPERIMENTS.items():
    counts = {c: len(members.get(c, ())) for c in classes}
    kept = [c for c in classes if counts[c] >= MIN_CLASS_SIZE]
    dropped = [c for c in classes if counts[c] < MIN_CLASS_SIZE]
    per_entity = {}
    for e in ENTITIES:
        cs = sorted(set(kept) & class_sets.get(e, set()))
        if cs:
            per_entity[e] = ','.join(cs)
    entities = list(per_entity)
    train_e, test_e = train_test_split(entities, test_size=TEST_SIZE, random_state=RANDOM_STATE)
    split_rows += [(experiment, e, 'train', per_entity[e]) for e in sorted(train_e)]
    split_rows += [(experiment, e, 'test', per_entity[e]) for e in sorted(test_e)]
    summary.append({'experiment': experiment, 'classes kept': len(kept), 'classes dropped': len(dropped),
                    'dropped': ', '.join(dropped), 'entities': len(entities),
                    'train': len(train_e), 'test': len(test_e),
                    'per-class counts': {c: sum(c in per_entity[e].split(',') for e in entities) for c in kept}})
entity_split = pd.DataFrame(split_rows, columns=['experiment', 'entity', 'split', 'classes'])
entity_split.to_csv(OUT / 'entity_split.tsv', sep='\t', header=True, index=False,
                    lineterminator='\n', quoting=csv.QUOTE_NONE, encoding='utf-8')
display(pd.DataFrame(summary).drop(columns='per-class counts'))

,experiment,classes kept,classes dropped,dropped,entities,train,test
0,Level-1,3,1,wordnet_body_of_water_109225146,8399,6719,1680
1,Level-2-Organizations,4,0,,428,342,86
2,Level-2-Persons,5,0,,3172,2537,635
3,Level-3-Artists,2,3,"wordnet_sculptor_110566072, wordnet_illustrato...",170,136,34


## 5. Outputs written

Files in `data/processed/fb15k-237/`. Sections 6 and 7 re-read them from disk.

In [7]:
for p in sorted(OUT.iterdir()):
    print(f'{p.name:20s} {p.stat().st_size:>10,d} bytes')

entity_classes.tsv    7,980,780 bytes
entity_split.tsv        676,367 bytes
test.tsv              1,499,033 bytes
train.tsv            21,005,177 bytes
valid.tsv             1,285,566 bytes


## 6. Acceptance checks

These cells read only the files on disk and recompute their expectations without reusing the processing code above, so they can disagree with it. Expected counts are literals from the spec. A failing assertion stops the notebook.

In [8]:
# 6.1 Downloads, triple files and entity universe
def sha256(path):
    return hashlib.sha256(Path(path).read_bytes()).hexdigest()

def read_rows(path):
    with open(path, encoding='utf-8', newline='') as f:
        text = f.read()
    return text.replace('\r\n', '\n').rstrip('\n').split('\n')

for name, (_, digest) in SOURCES.items():
    assert sha256(EXTERNAL / name) == digest, f'{name}: SHA256 mismatch'

dataset_entities = set()
for split in ['train', 'valid', 'test']:
    raw = (OUT / f'{split}.tsv').read_bytes()
    assert b'\r' not in raw and raw.endswith(b'\n'), f'{split}.tsv must be LF-terminated'
    raw.decode('utf-8')
    rows = [line.split('\t') for line in read_rows(OUT / f'{split}.tsv')]
    assert len(rows) == ROW_COUNTS[split], (split, len(rows))
    assert all(len(r) == 3 for r in rows), f'{split}.tsv must have exactly 3 columns, no header'
    # unchanged content and row order versus the download
    assert [r for r in rows] == [line.split('\t') for line in read_rows(EXTERNAL / f'{split}.txt')]
    for head, _, tail in rows:
        dataset_entities.update((head, tail))
    if split == 'train':
        committed = [line.split('\t') for line in read_rows(COMMITTED_TRAIN)]
        assert rows == committed, 'download differs from committed data/fb15k-237_train.tsv'
assert len(dataset_entities) == 14_541, len(dataset_entities)
print('triple files OK;', len(dataset_entities), 'entities')

triple files OK; 14541 entities


In [9]:
# 6.2 entity_classes.tsv
raw = (OUT / 'entity_classes.tsv').read_bytes()
assert b'\r' not in raw and raw.endswith(b'\n')
lines = raw.decode('utf-8').rstrip('\n').split('\n')
assert lines[0] == 'entity\tclass'
pairs = [tuple(l.split('\t')) for l in lines[1:]]
assert all(len(p) == 2 for p in pairs)
assert pairs == sorted(pairs) and len(pairs) == len(set(pairs)), 'must be sorted, one row per pair'
assert {e for e, _ in pairs} <= dataset_entities, 'entity_classes has entities outside the triples'
assert not any(c.startswith('<') or c.endswith('>') for _, c in pairs), 'class IDs carry angle brackets'
class_members = {}
for e, c in pairs:
    class_members.setdefault(c, set()).add(e)
print(len(pairs), 'pairs,', len({e for e, _ in pairs}), 'entities with a class,', len(class_members), 'classes')

212270

 pairs, 12615 entities with a class, 2260 classes


In [10]:
# 6.3 entity_split.tsv, per experiment
raw = (OUT / 'entity_split.tsv').read_bytes()
assert b'\r' not in raw and raw.endswith(b'\n')
lines = raw.decode('utf-8').rstrip('\n').split('\n')
assert lines[0] == 'experiment\tentity\tsplit\tclasses'
split_rows = [l.split('\t') for l in lines[1:]]
assert all(len(r) == 4 for r in split_rows)
assert {r[0] for r in split_rows} == set(EXPERIMENTS), 'experiments must be exactly the four of the spec'

for experiment, classes in EXPERIMENTS.items():
    rows = [r for r in split_rows if r[0] == experiment]
    entities = [r[1] for r in rows]
    assert len(entities) == len(set(entities)), f'{experiment}: duplicate entities'
    assert set(entities) <= dataset_entities
    train = {r[1] for r in rows if r[2] == 'train'}
    test = {r[1] for r in rows if r[2] == 'test'}
    assert {r[2] for r in rows} <= {'train', 'test'}
    assert not train & test, f'{experiment}: train/test overlap'
    n = len(rows)
    assert len(test) == (n + 4) // 5, f'{experiment}: |test| must be ceil(0.2 n)'  # ceil(n / 5) in integers
    # classes kept = experiment classes with >= 40 member entities, counted from entity_classes.tsv
    kept = sorted(c for c in classes if len(class_members.get(c, ())) >= MIN_CLASS_SIZE)
    assert kept, f'{experiment}: no class survives'
    # every entity with a kept class is in the split, with exactly its kept classes
    expected = {}
    for c in kept:
        for e in class_members[c]:
            expected.setdefault(e, []).append(c)
    assert {r[1]: r[3] for r in rows} == {e: ','.join(sorted(cs)) for e, cs in expected.items()}, \
        f'{experiment}: classes column disagrees with entity_classes.tsv'
    assert all(r[3] for r in rows), f'{experiment}: row without a class'
    per_class = {c: sum(c in r[3].split(',') for r in rows) for c in kept}
    assert all(v >= MIN_CLASS_SIZE for v in per_class.values()), f'{experiment}: class below {MIN_CLASS_SIZE}'
    # the sample assignment: sorted candidates, same random state, not stratified
    tr, te = train_test_split(sorted(entities), test_size=0.2, random_state=37)
    assert set(tr) == train and set(te) == test, f'{experiment}: split differs from train_test_split'
    print(f'{experiment}: OK ({n} entities, {len(train)}/{len(test)} train/test, kept {len(kept)}/{len(classes)} classes)')

Level-1: OK (8399 entities, 6719/1680 train/test, kept 3/4 classes)
Level-2-Organizations: OK (428 entities, 342/86 train/test, kept 4/4 classes)
Level-2-Persons: OK (3172 entities, 2537/635 train/test, kept 5/5 classes)
Level-3-Artists: OK (170 entities, 136/34 train/test, kept 2/5 classes)


## 7. Summary

Per experiment: classes kept and dropped (fewer than 40 entities with the class), entities in the entity split, train/test sizes and entities per kept class (train + test).

In [11]:
summary_df = pd.DataFrame(summary)
display(summary_df.drop(columns='per-class counts'))
for row in summary:
    print(row['experiment'])
    for c, n in row['per-class counts'].items():
        print(f'    {c:45s} {n:>6,d}')
    for c in row['dropped'].split(', ') if row['dropped'] else []:
        print(f'    {c:45s} dropped ({len(members.get(c, ()))} entities)')

,experiment,classes kept,classes dropped,dropped,entities,train,test
0,Level-1,3,1,wordnet_body_of_water_109225146,8399,6719,1680
1,Level-2-Organizations,4,0,,428,342,86
2,Level-2-Persons,5,0,,3172,2537,635
3,Level-3-Artists,2,3,"wordnet_sculptor_110566072, wordnet_illustrato...",170,136,34


Level-1
    wordnet_person_100007846                       4,446
    wordnet_organization_108008335                 2,360
    wordnet_product_104007894                      1,593
    wordnet_body_of_water_109225146               dropped (24 entities)
Level-2-Organizations
    wordnet_musical_organization_108246613           137
    wordnet_party_108256968                           42
    wordnet_enterprise_108056231                     170
    wordnet_nongovernmental_organization_108009834     82
Level-2-Persons
    wordnet_artist_109812338                       1,892
    wordnet_officeholder_110371450                   108
    wordnet_writer_110794014                       1,919
    wordnet_scientist_110560637                      115
    wordnet_politician_110450303                     245
Level-3-Artists
    wordnet_painter_110391653                         69
    wordnet_photographer_110426749                   110
    wordnet_sculptor_110566072                    dropped (15 entit